# Colab Worker — Corpo e Mente
Qwen3-VL-Embedding-2B
Runtime recomendado: T4 GPU

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest_asyncio \
    sentence-transformers langchain-text-splitters \
    opencv-python-headless pillow numpy \
    ultralytics

In [ ]:
import ultralytics
print("✅ ultralytics versión:", ultralytics.__version__)

# Verificar que las dependencias previas no se rompieron
import fastapi, uvicorn, cv2, numpy, PIL
from sentence_transformers import SentenceTransformer
print("✅ Dependencias previas intactas")

In [ ]:
import os
import cv2
import time
import subprocess
import sys
import numpy as np
import requests
from PIL import Image
from pyngrok import ngrok
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse

# Token de ngrok desde Colab Secrets (icono 🔑 a la izquierda)
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get('NGROK_TOKEN')
except Exception:
    NGROK_TOKEN = os.environ.get('NGROK_TOKEN', '')

if not NGROK_TOKEN:
    raise ValueError("❌ Configura NGROK_TOKEN en Colab Secrets (icono 🔑)")

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ Token ngrok configurado")

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ultralytics import YOLO

# --- 1. Estado inicial de VRAM ---
if torch.cuda.is_available():
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# --- 2. Cargar Qwen (ya estaba) ---
print("\nCargando Qwen3-VL-Embedding-2B...")
qwen_model = SentenceTransformer('Qwen/Qwen3-VL-Embedding-2B')
print("✅ Qwen cargado. Dim:", qwen_model.get_sentence_embedding_dimension())

if torch.cuda.is_available():
    print(f"VRAM usada tras Qwen: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras Qwen: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 3. Cargar YOLO26-pose ---
print("\nCargando YOLO26-pose...")
try:
    yolo_pose = YOLO('yolo26n-pose.pt')
    print("✅ YOLO26-pose cargado")
except Exception as e:
    print(f"❌ Error cargando YOLO26-pose: {e}")
    print("DETENTE Y REPORTA ESTE ERROR. No sustituyas por yolo11n-pose.pt sin autorización.")
    raise

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-pose: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-pose: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 4. Cargar YOLO26-depth (NUEVO) ---
print("\nCargando YOLO26-depth...")
try:
    yolo_depth = YOLO('yolo26n-depth.pt')
    print("✅ YOLO26-depth cargado")
except Exception as e:
    print(f"❌ Error cargando YOLO26-depth: {e}")
    print("DETENTE Y REPORTA ESTE ERROR. No sustituyas por otro modelo sin autorización.")
    raise

if torch.cuda.is_available():
    print(f"VRAM usada tras YOLO-depth: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"VRAM reservada tras YOLO-depth: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# --- 5. Verificar que Qwen sigue funcionando ---
print("\nVerificando que Qwen sigue operativo...")
test_emb = qwen_model.encode(["prueba de coexistencia"], normalize_embeddings=True)
print(f"✅ Qwen responde. Dim del vector: {len(test_emb[0])}")

print("\n✅ Coexistencia triple verificada: Qwen + YOLO26-pose + YOLO26-depth en la misma T4")

In [ ]:
app = FastAPI(title="Corpo e Mente Colab Worker")

@app.get("/health")
async def health():
    return {"status": "ok", "model": "Qwen3-VL-Embedding-2B"}

@app.post("/embed_text")
async def embed_text(texto: str = Form(...)):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500, chunk_overlap=200, length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(texto)
    embeddings = qwen_model.encode(chunks, normalize_embeddings=True)
    return JSONResponse({
        "chunks": chunks,
        "embeddings": embeddings.tolist(),
        "dimension": 2048,
        "chunk_size": 1500,
        "chunk_overlap": 200
    })

@app.post("/embed_video")
async def embed_video(video: UploadFile = File(...), fps_muestreo: int = 1):
    temp_path = f"/tmp/{video.filename}"
    with open(temp_path, "wb") as f:
        f.write(await video.read())

    cap = cv2.VideoCapture(temp_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    intervalo = max(1, int(fps / max(1, fps_muestreo)))
    frames, timestamps = [], []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % intervalo == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
            timestamps.append(round(idx / fps, 3))
        idx += 1
    cap.release()
    if os.path.exists(temp_path):
        os.remove(temp_path)

    if not frames:
        return JSONResponse({"error": "No se pudieron extraer frames"}, status_code=400)

    embeddings = qwen_model.encode(frames, normalize_embeddings=True)

    return JSONResponse({
        "embeddings": embeddings.tolist(),
        "timestamps": timestamps,
        "total_frames": len(frames),
        "dimension": 2048
    })

print("✅ FastAPI definido con /health, /embed_text, /embed_video")

In [ ]:
# 1. Levantar uvicorn en un hilo, forzando IPv4
def _run_uvicorn():
    import asyncio
    from uvicorn.config import Config
    from uvicorn.server import Server

    # Create a uvicorn Config object
    config = Config(app, host="127.0.0.1", port=8000, log_level="warning")
    # Instantiate the server
    server = Server(config=config)
    # Run the server's serve method using the patched asyncio.run,
    # which avoids uvicorn's internal call to asyncio_run with loop_factory.
    asyncio.run(server.serve())

threading.Thread(target=_run_uvicorn, daemon=True).start()

# 2. Esperar a que uvicorn escuche localmente
print("⏳ Esperando a que uvicorn levante...")
uvicorn_ok = False
for i in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print(f"✅ uvicorn listo en {i+1}s: {r.json()}")
            uvicorn_ok = True
            break
    except Exception:
        pass
    time.sleep(1)

if not uvicorn_ok:
    raise RuntimeError("❌ uvicorn no respondió en 30s. Revisa errores de importación.")

# 3. Conectar ngrok apuntando a IPv4 explícita
ngrok.kill()  # limpiar túneles previos
ngrok_tunnel = ngrok.connect("http://127.0.0.1:8000", bind_tls=True)

print("=" * 60)
print(f"COLAB_TUNNEL_URL = {ngrok_tunnel.public_url}")
print("Copia esta URL en tu backend/.env como COLAB_TUNNEL_URL")
print("=" * 60)

# 4. Probar el túnel público
time.sleep(2)
try:
    r = requests.get(f"{ngrok_tunnel.public_url}/health", timeout=15)
    print(f"🌐 Túnel público → {r.status_code}: {r.text}")
except Exception as e:
    print(f"⚠️ No se pudo probar el túnel público: {e}")
    print("Pero uvicorn está corriendo localmente. Prueba desde tu navegador.")